# Prompt para diseño de esta clase    

Dame el diseño de una clase, bien completa, sobre: "Archivos de Datos .csv y Bases de Datos .db de SQL", de 3 horas para la formación de estudiantes del segundo semestre de la carrera de Matemáticas en Caucasia - Antioquia (Colombia). Donde los temas a desarrollar sean:  

1. Creación de archivos de datos .csv a partit de DataFrame de Pandas. 
2. Creación de bases de datos sql con sqlite. 
3. Acceso a bases de datos sql con Pandas.
4. Consultas a la base de datos utilizando la sentencia SELECT

Todo con una orientación hacia su utilización en ciencia de datos con ejemplos de la Teoría Elemental de Números y pensando en desarrollar las habilidades computacionales de los estudiantes para el curso de Métodos Numéricos I que verán en dos semestres. Esta es una clase para estudiantes que ya tienen los conceptos adquiricos de Tipos de datos, Estructuras de datos incorporadas, Operadores, condicionales, ciclos, funciones de usuario de Python, NumPy, Pandas y Matplotlib. Y esto teniendo en cuenta que el siguiente tema en este curso es "Análisis Descriptivo de Datos". 

# Diseño de Clase: Archivos de Datos .csv y Bases de Datos .db de SQL

**Programa:** Matemáticas | **Semestre:** Segundo | **Curso:** (Preparación para Métodos Numéricos I)  
**Ubicación:** Caucasia, Antioquia, Colombia | **Duración:** 3 horas (180 minutos)  
**Temática:** Ciencia de Datos con orientación a Teoría Elemental de Números  
**Prerrequisitos:** Python (estructuras, condicionales, ciclos, funciones), NumPy, Pandas, Matplotlib.

---

## 1. Objetivos de Aprendizaje

Al finalizar la clase, el estudiante estará en capacidad de:
1. **Exportar** DataFrames de Pandas a archivos planos `.csv` controlando la codificación y el índice, para garantizar la persistencia de datos numéricos.

2. **Crear** bases de datos relacionales ligeras `.db` utilizando `sqlite3`, definiendo esquemas de tablas con tipos de datos adecuados.

3. **Conectar** y consultar bases de datos SQLite desde Pandas mediante `pd.read_sql`, integrando la potencia de SQL con la manipulación vectorial de DataFrames.

4. **Formular** consultas `SELECT` con filtros (`WHERE`), ordenamientos (`ORDER BY`) y agregaciones (`GROUP BY`), aplicadas a la resolución de problemas de Teoría de Números (primalidad, divisibilidad, funciones aritméticas).

## 2. Contextualización y Justificación

En el curso de **Métodos Numéricos I** (dos semestres adelante), los estudiantes enfrentarán problemas donde la gestión eficiente de datos es crucial: por ejemplo, almacenar los coeficientes de un sistema de ecuaciones, registrar los iterados de un método (bisección, Newton-Raphson) o manejar mallas de interpolación. El uso de `.csv` y `.db` permite **desacoplar la captura de datos del análisis descriptivo** que verán a continuación.

La **Teoría Elemental de Números** se elige como hilo conductor porque:
- Los datos son **discretos y verificables** (primos, divisores, coprimalidad).
- Permite generar **conjuntos de datos sintéticos** de manera programática sin depender de archivos externos.
- Los estudiantes de matemáticas ya poseen intuición sobre estas propiedades, lo que reduce la carga cognitiva hacia la sintaxis de SQL/Pandas.

---

## 3. Contenidos y Distribución del Tiempo

| Bloque | Tema | Tiempo | Actividad |
| :--- | :--- | :--- | :--- |
| **1** | Creación de `.csv` desde Pandas | 40 min | Demostración + Ejercicio guiado |
| **2** | Creación de Base de Datos SQLite | 35 min | Explicación + Creación de tabla |
| **3** | Acceso a SQL con Pandas | 25 min | Integración de flujos |
| **4** | Consultas SELECT | 50 min | Taller dirigido (Teoría de Números) |
| **5** | Cierre y Síntesis | 30 min | Ejercicio integrador y conclusiones |

---

## 4. Desarrollo Detallado de la Clase

### Bloque 1: Creación de archivos de datos `.csv` a partir de DataFrames de Pandas (40 min)

**Concepto Clave:** El formato CSV (Comma-Separated Values) es el estándar *de facto* para intercambiar datos tabulares. En ciencia de datos, es vital controlar la escritura para no perder información (como el índice numérico).

**Ejemplo con Teoría de Números:**  
Vamos a generar una tabla con los primeros $N$ números enteros y sus propiedades básicas: si es primo, su número de divisores ($\tau(n)$) y la suma de sus divisores ($\sigma(n)$).

In [ ]:
import pandas as pd
import numpy as np

# Generación de datos con orientación matemática
N = 100
datos = []
for n in range(1, N + 1):
    divisores = [d for d in range(1, n + 1) if n % d == 0]
    es_primo = len(divisores) == 2
    datos.append({
        'n': n,
        'es_primo': es_primo,
        'num_divisores': len(divisores),
        'suma_divisores': sum(divisores)
    })

In [ ]:
df_numeros = pd.DataFrame(datos)

In [ ]:
df_numeros.head(10)

**Sintaxis Crítica para la Exportación:**  
La función `df.to_csv()` es la herramienta principal. Para el curso, enfatizamos dos parámetros:
- `index=False`: Evita que Pandas escriba el índice numérico (0,1,2...) como una columna extra, lo cual es indeseado al compartir datos.
- `encoding='utf-8'`: Asegura compatibilidad con tildes y caracteres especiales en español (útil si los nombres de columnas llevan texto).

In [ ]:
# Exportación
df_numeros.to_csv('propiedades_numeros.csv', index=False, encoding='utf-8')

**Verificación:** El estudiante debe abrir el archivo (o usar `pd.read_csv`) para confirmar que la estructura se mantiene.

In [ ]:
# Verificación
df_verif = pd.read_csv('propiedades_numeros.csv')

In [ ]:
print(df_verif.head())

In [ ]:
print(df_verif.shape)

---

### Bloque 2: Creación de Bases de Datos SQL con SQLite (35 min)

**Concepto Clave:** A diferencia del CSV (que es un archivo de texto plano), una base de datos `.db` es un **motor relacional** que permite consultas eficientes, integridad de tipos y relaciones entre tablas. `sqlite3` viene integrado en Python, sin necesidad de instalar servidores externos.

**Flujo de Creación:**
1. Conectar (crear el archivo si no existe).
2. Crear un cursor (objeto para ejecutar comandos).
3. Ejecutar `CREATE TABLE` con tipos de datos (`INTEGER`, `REAL`, `TEXT`).
4. Insertar datos (preferiblemente desde el DataFrame ya creado para mantener consistencia).
5. Confirmar (`commit`) y cerrar.

In [ ]:
import sqlite3

# 1. Conexión
conn = sqlite3.connect('teoria_numeros.db')
cursor = conn.cursor()

# 2. Creación de tabla
cursor.execute('''
    CREATE TABLE IF NOT EXISTS numeros (
        n INTEGER PRIMARY KEY,
        es_primo INTEGER,
        num_divisores INTEGER,
        suma_divisores INTEGER
    )
''')

# 3. Inserción masiva desde el DataFrame (usando executemany)
datos_insercion = df_numeros.values.tolist()
cursor.executemany('''
    INSERT OR REPLACE INTO numeros (n, es_primo, num_divisores, suma_divisores)
    VALUES (?, ?, ?, ?)
''', datos_insercion)

conn.commit()
conn.close()
print("Base de datos creada exitosamente.")

**Nota Didáctica:** Se recomienda usar `INSERT OR REPLACE` para evitar errores de clave duplicada si el estudiante ejecuta el script varias veces.

---

### Bloque 3: Acceso a Bases de Datos SQL con Pandas (25 min)

**Concepto Clave:** Pandas puede actuar como un "puente" entre Python y SQL. En lugar de usar el cursor y traer listas de tuplas (que luego habría que convertir), `pd.read_sql()` ejecuta una consulta y devuelve **directamente un DataFrame**.

**Flujo de Trabajo:**

In [ ]:
# 1. Reconectar (o mantener la conexión viva)
conn = sqlite3.connect('teoria_numeros.db')

# 2. Consulta directa a DataFrame
df_primos = pd.read_sql('SELECT * FROM numeros WHERE es_primo = 1', conn)

print(df_primos)
conn.close()

**Ventaja Competitiva:** El estudiante ahora puede aplicar toda la potencia de Pandas (`.describe()`, `.plot()`, agrupaciones) sobre los resultados de una consulta SQL, sin salir del entorno de Python. Esto es exactamente lo que se hace en flujos profesionales de ciencia de datos: SQL para extracción, Pandas para transformación y Matplotlib para visualización.

---

### Bloque 4: Consultas a la Base de Datos utilizando la Sentencia SELECT (50 min)

**Estructura de la Sentencia (Sintaxis Básica):**
```sql
SELECT columnas
FROM tabla
WHERE condición
ORDER BY columna [ASC|DESC]
LIMIT cantidad;
```

> **Nota:** como la conexión se cerró al final del Bloque 3, se vuelve a abrir antes de los ejercicios y se cierra al terminar el bloque.

In [ ]:
conn = sqlite3.connect('teoria_numeros.db')

**Ejercicios Dirigidos (Teoría de Números):**

**Ejercicio 1: Filtrado Básico (WHERE)**  
*Objetivo:* Encontrar los números primos gemelos (pares de primos con diferencia 2) en el rango 1-100.  
*Nota:* SQL no tiene bucles, pero podemos hacer una **autoconexión** (self-join) para comparar la tabla consigo misma.

In [ ]:
query = '''
    SELECT a.n AS primo_menor, b.n AS primo_mayor, (b.n - a.n) AS diferencia
    FROM numeros a
    JOIN numeros b ON b.n = a.n + 2
    WHERE a.es_primo = 1 AND b.es_primo = 1
    ORDER BY a.n;
'''
df_gemelos = pd.read_sql(query, conn)
print(df_gemelos)

*Reflexión Matemática:* ¿Cuántos pares de gemelos hay antes de 100? ¿Qué nos dice esto sobre la distribución de primos?

**Ejercicio 2: Agregación (GROUP BY)**  
*Objetivo:* Calcular el promedio de divisores de los números primos vs. los compuestos.

In [ ]:
query = '''
    SELECT es_primo, AVG(num_divisores) AS promedio_divisores, COUNT(*) AS cantidad
    FROM numeros
    GROUP BY es_primo;
'''
df_agrupado = pd.read_sql(query, conn)
print(df_agrupado)

*Resultado esperado:* Para primos, el promedio debe ser 2.0; para compuestos, mayor.

**Ejercicio 3: Ordenamiento y Límite (ORDER BY + LIMIT)**  
*Objetivo:* Encontrar el número menor a 100 con la mayor suma de divisores (número altamente compuesto).

In [ ]:
query = '''
    SELECT n, suma_divisores
    FROM numeros
    WHERE n <= 100
    ORDER BY suma_divisores DESC
    LIMIT 5;
'''
df_top = pd.read_sql(query, conn)
print(df_top)

**Discusión en Clase:** ¿Qué relación tiene la suma de divisores con los números perfectos (ej. 6, 28)? ¿Cómo podríamos usar SQL para buscarlos?

In [ ]:
conn.close()

---

### Bloque 5: Cierre y Ejercicio Integrador (30 min)

**Reto Final (Evaluación Formativa):**  
El estudiante debe escribir un script que:
1. Genere un DataFrame con los números del 1 al 200 y su factorización (o al menos el número de factores primos distintos).
2. Guarde ese DataFrame en un archivo `factores.csv`.
3. Cree (o actualice) una base de datos `factores.db`.
4. Utilice `pd.read_sql` para extraer los números que tienen exactamente 3 divisores (que matemáticamente son los cuadrados de primos: $p^2$).

In [ ]:
# Tu solución aquí

**Pregunta de Cierre:**  
*"Si mañana necesitan procesar los 10,000,000 de iterados de un método numérico para encontrar raíces, ¿usarían un CSV o una base de datos SQL? ¿Por qué?"*  
(Respuesta esperada: SQLite es más eficiente para búsquedas y filtrados sobre grandes volúmenes, mientras que CSV es útil para datos pequeños o interoperabilidad simple).

---

## 5. Recursos y Materiales

- **Entorno:** Jupyter Notebook o Google Colab (para que los estudiantes sin computador potente puedan participar).
- **Archivos:** `propiedades_numeros.csv`, `teoria_numeros.db` (generados en clase).
- **Referencias Bibliográficas:**
    - Documentación oficial de Pandas: `to_csv` y `read_sql`.
    - Documentación de Python: `sqlite3`.
    - Material de apoyo: Guía rápida de SQL (SELECT, WHERE, GROUP BY).

---

## 6. Evaluación

| Criterio | Indicador | Instrumento |
| :--- | :--- | :--- |
| **Uso de CSV** | Exporta correctamente sin el índice y con codificación adecuada. | Revisión del archivo generado. |
| **Uso de SQLite** | Crea la tabla con tipos de datos coherentes e inserta sin errores. | Verificación de la base de datos. |
| **Integración Pandas-SQL** | Usa `pd.read_sql` de forma autónoma. | Ejercicio integrador. |
| **Pensamiento Matemático** | Interpreta los resultados de las consultas en términos de propiedades numéricas. | Preguntas de reflexión en clase. |

---

## 7. Adaptaciones para Caucasia (Contexto)

- **Conectividad:** Si el internet es inestable, se sugiere que el docente lleve los scripts pre-cargados en un USB y los estudiantes trabajen en modo local con Python instalado (Anaconda).
- **Ejemplos Cotidianos:** Si bien la Teoría de Números es abstracta, se puede mencionar que la **criptografía RSA** (usada en bancos y WhatsApp) se basa en la dificultad de factorizar números grandes, lo cual conecta directamente con las tablas de primos y divisores creadas en clase.

# Ejercicios para tarea  

# Taller Evaluativo: 4 Problemas Alineados a los Objetivos de Aprendizaje

**Curso:** Fundamentos de Programación | **Semestre:** Segundo | **Programa:** Matemáticas
**Contexto:** Caucasia, Antioquia | **Temática:** CSV, SQLite, Pandas y SELECT con ejemplos en Teoría Elemental de Números

## Orientaciones Generales

- **Modalidad:** Trabajo individual o en parejas (máximo 2 estudiantes).

- **Tiempo estimado:** 30 minutos (puede asignarse como trabajo fuera de clase).

- **No hay que Entregar:** Un notebook de Jupyter (`.ipynb`) o script `.py` con las soluciones, más los archivos generados (`*.csv` y `*.db`).

- **Recomendación:** Cada problema debe resolverse en una celda independiente y comentada, evidenciando el razonamiento matemático detrás de la consulta.

> **Nota pedagógica:** Los 4 problemas están diseñados en dificultad creciente y cada uno apunta explícitamente a uno de los objetivos de la clase, aunque el Problema 4 los integra todos.



## Problema 1 — Creación de archivos `.csv` desde un DataFrame de Pandas

**Objetivo evaluado:** Exportar DataFrames a `.csv` controlando codificación e índice.

### Enunciado
En Teoría Elemental de Números, la **función $\varphi$ de Euler** (función indicatriz) cuenta cuántos enteros positivos menores o iguales que $n$ son coprimos con $n$. Es decir:
$$
\varphi(n) = \#\{k \in \{1,\dots,n\} : \gcd(k,n)=1\}
$$

**Tareas:**
1. Construya un DataFrame llamado `df_phi` con los números $n = 1, 2, \dots, 150$ y las columnas:
   - `n`: el número entero.
   - `phi_n`: el valor de $\varphi(n)$.
   - `es_primo`: booleano (`True`/`False`) según si $n$ es primo.
   - `cociente_phi_n`: el valor $\varphi(n)/n$ (útil para estudiar la densidad de coprimos).

In [8]:
import numpy as np 
import pandas as pd 
import math
N = 150
datos = []
for n in range(1, N+1):
    phi_n = sum(1 for k in range(1, n+1) if math.gcd(k,n) == 1)
    es_primo = len([d  for d in range(1,n+1) if n%d==0]) == 2
    cociente_phi_n = phi_n/n
    datos.append({
        'n': n,
        'phi_n':phi_n,
        'es_primo':es_primo,
        'cociente_phi_n': cociente_phi_n
    })

df = pd.DataFrame(datos)
df[51:101] 

,n,phi_n,es_primo,cociente_phi_n
51,52,24,False,0.461538
52,53,52,True,0.981132
53,54,18,False,0.333333
54,55,40,False,0.727273
55,56,24,False,0.428571
56,57,36,False,0.631579
57,58,28,False,0.482759
58,59,58,True,0.983051
59,60,16,False,0.266667
60,61,60,True,0.983607


2. Exporte el DataFrame a un archivo llamado `funcion_phi.csv` **sin incluir el índice** y **con codificación UTF-8**.

In [6]:
df.to_csv('funcion_phi.csv', index = False, encoding = 'utf-8') 

3. Verifique la exportación leyendo nuevamente el archivo con `pd.read_csv` y comprobando que tiene exactamente 150 filas y 4 columnas.

In [7]:
pd.read_csv('funcion_phi.csv')

,n,phi_n,es_primo,cociente_phi_n
0,1,1,False,1.000000
1,2,1,True,0.500000
2,3,2,True,0.666667
3,4,2,False,0.500000
4,5,4,True,0.800000
...,...,...,...,...
145,146,72,False,0.493151
146,147,84,False,0.571429
147,148,72,False,0.486486
148,149,148,True,0.993289


4. **Reflexión escrita (3–5 líneas):** ¿Qué patrón observa en `cociente_phi_n` para los números primos? ¿Por qué ocurre esto matemáticamente?

### Pistas
- Use `math.gcd` para calcular el máximo común divisor.
- Recuerde que `np.arange(1, 151)` genera los enteros del 1 al 150.
- Revise la diferencia entre `index=True` (por defecto) e `index=False`.

### Criterios de Evaluación
| Aspecto | Puntaje |
| :--- | :--- |
| Cálculo correcto de $\varphi(n)$ para los 150 valores | 1.0 |
| Exportación con `index=False` y `encoding='utf-8'` | 0.5 |
| Verificación mediante relectura | 0.5 |
| Reflexión matemática coherente | 1.0 |
| **Total** | **3.0** |

---

## Problema 2 — Creación de una base de datos SQL con SQLite
**🎯 Objetivo evaluado:** Crear bases de datos `.db` definiendo esquemas con tipos de datos adecuados.

### Enunciado
Los **números perfectos** son aquellos que son iguales a la suma de sus divisores propios (por ejemplo, \(6 = 1+2+3\), \(28 = 1+2+4+7+14\)). Vamos a construir una base de datos para estudiarlos.

**Tareas:**
1. Genere un DataFrame `df_perfectos` con los números \(n = 1, \dots, 500\) y las columnas:
   - `n` (entero)
   - `suma_divisores_propios` (entero)
   - `es_perfecto` (booleano → almacenar como 0/1)
   - `es_abundante` (booleano: suma de divisores propios > n)
   - `es_deficiente` (booleano: suma de divisores propios < n)
2. Cree una base de datos llamada `numeros_especiales.db` con una tabla `perfectos` cuyo esquema sea:
   ```sql
   CREATE TABLE perfectos (
       n INTEGER PRIMARY KEY,
       suma_divisores_propios INTEGER,
       es_perfecto INTEGER,
       es_abundante INTEGER,
       es_deficiente INTEGER
   );
   ```
3. Inserte todos los registros del DataFrame en la tabla. Use `INSERT OR REPLACE` para que el script sea **reejecutable** sin errores de clave duplicada.
4. Confirme la inserción consultando el número total de filas de la tabla con `SELECT COUNT(*) FROM perfectos`.
5. **Reflexión escrita:** Mencione al menos 3 números perfectos que aparezcan en la base de datos y explique por qué se les llama "perfectos".

### Pistas
- Use `conn = sqlite3.connect('numeros_especiales.db')`.
- No olvide `conn.commit()` antes de cerrar.
- Puede convertir booleanos a enteros con `.astype(int)`.

### Criterios de Evaluación
| Aspecto | Puntaje |
| :--- | :--- |
| DataFrame correcto con las 5 columnas | 1.0 |
| Creación de la tabla con tipos de datos coherentes | 0.5 |
| Inserción completa y reejecutable (`INSERT OR REPLACE`) | 1.0 |
| Verificación con `COUNT(*)` | 0.5 |
| Reflexión matemática sobre números perfectos | 1.0 |
| **Total** | **4.0** |

---

## Problema 3 — Acceso a bases de datos SQL con Pandas
**🎯 Objetivo evaluado:** Conectar y consultar SQLite desde Pandas con `pd.read_sql`, integrando flujos de trabajo.

### Enunciado
En ciencia de datos es común **extraer** información de una base de datos y luego **transformarla** con Pandas. En este problema trabajará con la base de datos `numeros_especiales.db` creada en el Problema 2 (o con la que le entregue el docente).

**Tareas:**
1. Conéctese a la base de datos y cargue **toda la tabla** `perfectos` en un DataFrame llamado `df_db` usando `pd.read_sql`.
2. Verifique que `df_db.shape` coincide con el DataFrame original del Problema 2.
3. Con el DataFrame `df_db`, realice las siguientes operaciones **usando Pandas** (no SQL):
   - a) Cuente cuántos números son perfectos, abundantes y deficientes (use `value_counts` o `sum`).
   - b) Calcule el promedio de `suma_divisores_propios` para los números primos (recuerde que para un primo \(p\), la suma de divisores propios es 1).
   - c) Grafique con Matplotlib un histograma de `suma_divisores_propios` diferenciando por tipo (perfecto / abundante / deficiente).
4. **Reflexión escrita:** ¿Qué ventaja tiene usar `pd.read_sql` frente a usar el cursor de `sqlite3` y traer listas de tuplas? ¿En qué situaciones preferiría SQL y en cuáles Pandas?

### Pistas
- Importe `matplotlib.pyplot as plt`.
- Use `df_db.groupby('es_perfecto')['suma_divisores_propios'].mean()` para agrupaciones rápidas.
- Recuerde cerrar la conexión con `conn.close()`.

### Criterios de Evaluación
| Aspecto | Puntaje |
| :--- | :--- |
| Uso correcto de `pd.read_sql` | 1.0 |
| Verificación de dimensiones | 0.5 |
| Cálculos con Pandas (conteos, promedios) | 1.0 |
| Gráfico claro con leyendas y título | 1.0 |
| Reflexión sobre ventajas Pandas vs SQL | 0.5 |
| **Total** | **4.0** |

---

## Problema 4 — Consultas SELECT aplicadas a la Teoría de Números
**🎯 Objetivo evaluado:** Formular consultas `SELECT` con `WHERE`, `ORDER BY`, `GROUP BY` y agregaciones.

### Enunciado
Vamos a explotar la base de datos `numeros_especiales.db` con **consultas SQL puras** (sin usar Pandas para filtrar, aunque sí para visualizar los resultados). El objetivo es descubrir propiedades numéricas ocultas en los datos.

**Tareas:**

**a) Filtrado simple (WHERE):**
Encuentre todos los números perfectos menores o iguales a 500. ¿Cuántos hay? ¿Cuáles son?

```sql
SELECT n, suma_divisores_propios
FROM perfectos
WHERE es_perfecto = 1 AND n <= 500;
```

**b) Ordenamiento y límite (ORDER BY + LIMIT):**
Encuentre los **5 números más abundantes** (mayor exceso de `suma_divisores_propios` sobre \(n\)), ordenados de mayor a menor.

**c) Agregación (GROUP BY):**
Calcule, para cada categoría (perfecto, abundante, deficiente), cuántos números hay y el promedio de su `suma_divisores_propios`. Use `CASE` o los campos booleanos ya existentes.

**d) Consulta con subconsulta:**
Encuentre los números \(n\) cuya `suma_divisores_propios` sea **mayor que el promedio** de todas las sumas de divisores propios de la tabla.

**e) Reto adicional (opcional):**
Encuentre los **primos gemelos** \((p, p+2)\) dentro de la tabla (recuerde que un primo \(p\) tiene `suma_divisores_propios = 1`). Use una autoconexión (self-join).

```sql
SELECT a.n AS primo_menor, b.n AS primo_mayor
FROM perfectos a
JOIN perfectos b ON b.n = a.n + 2
WHERE a.suma_divisores_propios = 1
  AND b.suma_divisores_propios = 1;
```

**f) Reflexión escrita (5–7 líneas):**
Explique con sus palabras qué hace cada consulta y qué propiedad matemática está explorando. Mencione al menos una conexión entre estos resultados y algún concepto que haya visto en Teoría de Números.

### Pistas
- Recuerde la estructura: `SELECT ... FROM ... WHERE ... GROUP BY ... ORDER BY ... LIMIT ...`.
- Para el literal (b), puede usar `ORDER BY (suma_divisores_propios - n) DESC LIMIT 5`.
- Para el literal (d), combine: `WHERE suma_divisores_propios > (SELECT AVG(suma_divisores_propios) FROM perfectos)`.

### Criterios de Evaluación
| Aspecto | Puntaje |
| :--- | :--- |
| (a) Filtrado con `WHERE` correcto | 0.5 |
| (b) Ordenamiento con `ORDER BY` + `LIMIT` | 0.5 |
| (c) Agregación con `GROUP BY` | 1.0 |
| (d) Subconsulta bien formulada | 1.0 |
| (e) Autoconexión (reto opcional) | +0.5 (bonus) |
| (f) Reflexión matemática integradora | 1.0 |
| **Total** | **4.0 (+0.5 bonus)** |

---

## Rúbrica Global del Taller

| Problema | Objetivo Asociado | Puntaje |
| :--- | :--- | :--- |
| 1 | Creación de `.csv` desde Pandas | 3.0 |
| 2 | Creación de base de datos SQLite | 4.0 |
| 3 | Acceso a SQL con Pandas | 4.0 |
| 4 | Consultas SELECT | 4.0 |
| **Total** | | **15.0** |

**Escala de valoración sugerida (Colombia):**
- **Superior (4.6–5.0):** 13.5–15.0 puntos.
- **Alto (4.0–4.5):** 12.0–13.4 puntos.
- **Básico (3.0–3.9):** 9.0–11.9 puntos.
- **Bajo (<3.0):** Menos de 9.0 puntos.

---

## Recomendaciones para el Docente

1. **Retroalimentación inmediata:** Comparta al final del taller una solución de referencia en el tablero o proyector, especialmente para el Problema 4, donde la sintaxis SQL puede generar confusión.
2. **Conexión con el siguiente tema:** Cierre el taller mencionando que los DataFrames obtenidos con `pd.read_sql` son la entrada natural para el **Análisis Descriptivo de Datos** (medias, medianas, histogramas) que verán en la próxima sesión.
3. **Contexto local:** Anímelos a comentar cómo la criptografía RSA (usada en la banca y en WhatsApp, servicios presentes en Caucasia) depende de la factorización de números grandes, lo cual conecta directamente con las tablas de divisores y primos trabajadas.
4. **Adaptación sin internet:** Entregue los archivos `funcion_phi.csv` y `numeros_especiales.db` en un USB para estudiantes con conectividad limitada, y permítales trabajar en Python local (Anaconda) en lugar de Colab.

---

## Anexo: Solución de Referencia Rápida

```python
# --- PROBLEMA 1 ---
import pandas as pd, numpy as np, math

datos_phi = []
for n in range(1, 151):
    coprimos = sum(1 for k in range(1, n+1) if math.gcd(k, n) == 1)
    divisores = [d for d in range(1, n+1) if n % d == 0]
    datos_phi.append({
        'n': n,
        'phi_n': coprimos,
        'es_primo': len(divisores) == 2,
        'cociente_phi_n': coprimos / n
    })
df_phi = pd.DataFrame(datos_phi)
df_phi.to_csv('funcion_phi.csv', index=False, encoding='utf-8')
verificacion = pd.read_csv('funcion_phi.csv')
assert verificacion.shape == (150, 4)

In [ ]:
# --- PROBLEMA 2 ---
import sqlite3
datos_perf = []
for n in range(1, 501):
    divs = [d for d in range(1, n) if n % d == 0]
    s = sum(divs)
    datos_perf.append({
        'n': n,
        'suma_divisores_propios': s,
        'es_perfecto': int(s == n),
        'es_abundante': int(s > n),
        'es_deficiente': int(s < n)
    })
df_perf = pd.DataFrame(datos_perf)

conn = sqlite3.connect('numeros_especiales.db')
cur = conn.cursor()
cur.execute('''CREATE TABLE IF NOT EXISTS perfectos (
    n INTEGER PRIMARY KEY,
    suma_divisores_propios INTEGER,
    es_perfecto INTEGER,
    es_abundante INTEGER,
    es_deficiente INTEGER)''')
cur.executemany('INSERT OR REPLACE INTO perfectos VALUES (?,?,?,?,?)',
                df_perf.values.tolist())
conn.commit()

In [ ]:
# --- PROBLEMA 3 ---
df_db = pd.read_sql('SELECT * FROM perfectos', conn)
print(df_db.shape)
print(df_db[['es_perfecto','es_abundante','es_deficiente']].sum())
conn.close()

In [ ]:
# --- PROBLEMA 4 ---
conn = sqlite3.connect('numeros_especiales.db')
q_a = 'SELECT n, suma_divisores_propios FROM perfectos WHERE es_perfecto=1 AND n<=500'
q_b = 'SELECT n, (suma_divisores_propios - n) AS exceso FROM perfectos ORDER BY exceso DESC LIMIT 5'
q_c = '''SELECT
    CASE WHEN es_perfecto=1 THEN 'perfecto'
         WHEN es_abundante=1 THEN 'abundante'
         ELSE 'deficiente' END AS tipo,
    COUNT(*) AS cantidad,
    AVG(suma_divisores_propios) AS promedio
    FROM perfectos GROUP BY tipo'''
q_d = 'SELECT n FROM perfectos WHERE suma_divisores_propios > (SELECT AVG(suma_divisores_propios) FROM perfectos)'
q_e = '''SELECT a.n AS primo_menor, b.n AS primo_mayor
         FROM perfectos a JOIN perfectos b ON b.n = a.n + 2
         WHERE a.suma_divisores_propios = 1 AND b.suma_divisores_propios = 1'''

for q in [q_a, q_b, q_c, q_d, q_e]:
    print(f"\n--- {q[:40]}... ---")
    print(pd.read_sql(q, conn))
conn.close()
```

---

Este taller garantiza que cada objetivo de la clase quede evaluado con evidencia concreta y verificable, y mantiene el hilo conductor de la **Teoría Elemental de Números** que conecta con los futuros contenidos de **Métodos Numéricos I** y con el siguiente tema de **Análisis Descriptivo de Datos**.